# EA2 - Actividad 2.4: Machine Learning con Spark MLlib

## Objetivos
- Entender los conceptos fundamentales de Spark MLlib
- Construir un pipeline de Machine Learning completo
- Aplicar feature engineering con Transformers
- Entrenar modelos de clasificacion (DecisionTree, RandomForest)
- Evaluar el rendimiento de los modelos

## Conceptos Clave

### Spark MLlib

MLlib es la libreria de Machine Learning de Spark. Permite entrenar modelos sobre datos distribuidos, escalando a millones de registros. Usa la API de DataFrames (`spark.ml`).

### Componentes Principales

| Componente | Descripcion | Ejemplo |
|------------|-------------|----------|
| **Transformer** | Transforma un DataFrame en otro | `StringIndexer`, `VectorAssembler` |
| **Estimator** | Algoritmo que se entrena sobre datos | `DecisionTreeClassifier`, `RandomForestClassifier` |
| **Pipeline** | Cadena de Transformers + Estimators | `Pipeline(stages=[...])` |
| **Evaluator** | Mide el rendimiento del modelo | `MulticlassClassificationEvaluator` |

### Feature Engineering

Los modelos de MLlib requieren que las features esten en un **vector numerico**. El proceso tipico es:

1. **StringIndexer:** Convierte columnas categoricas (texto) a indices numericos
2. **Binarizer:** Convierte una columna numerica a binaria (0/1) segun un umbral
3. **VectorAssembler:** Combina multiples columnas en un solo vector de features

### Pipeline ML

Un Pipeline encadena todos los pasos de transformacion y entrenamiento:
```
Datos -> StringIndexer -> Binarizer -> VectorAssembler -> Clasificador -> Predicciones
```
Ventajas:
- Reproducibilidad: mismo pipeline para train y test
- Facilidad: un solo `fit()` entrena todo
- Prevencion de data leakage

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, Binarizer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("ml_spark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Cargar y Explorar Datos

Usaremos el dataset de vuelos para predecir si un vuelo tendra retraso. Primero exploramos la distribucion de retrasos.

In [ ]:
# Leer datos de vuelos
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

print(f"Total de registros: {df.count()}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

In [ ]:
# Explorar distribucion de retrasos
df.select("DEPARTURE_DELAY").describe().show()

In [ ]:
# Ver distribucion: vuelos retrasados vs a tiempo (umbral: 15 minutos)
df.groupBy(
    F.when(F.col("DEPARTURE_DELAY") > 15, "Retrasado") \
     .otherwise("A tiempo") \
     .alias("estado")
).count().show()

## 2. Preparar Datos

Antes de entrenar el modelo, necesitamos limpiar los datos eliminando filas con valores nulos en las columnas clave.

In [ ]:
# Eliminar filas con valores nulos en columnas clave
df_clean = df.dropna(subset=[
    "DEPARTURE_DELAY", "AIRLINE", "ORIGIN_AIRPORT", 
    "DESTINATION_AIRPORT", "DISTANCE", "MONTH", "DAY_OF_WEEK"
])

print(f"Registros originales: {df.count()}")
print(f"Registros despues de limpieza: {df_clean.count()}")
print(f"Registros eliminados: {df.count() - df_clean.count()}")

## 3. Feature Engineering: StringIndexer

Los modelos de ML necesitan datos numericos. `StringIndexer` convierte columnas de texto (como AIRLINE) en indices numericos.

In [ ]:
# Codificar variables categoricas con StringIndexer
indexer_airline = StringIndexer(inputCol="AIRLINE", outputCol="airline_idx")
indexer_origin = StringIndexer(inputCol="ORIGIN_AIRPORT", outputCol="origin_idx")

# Ejemplo: ver como funciona StringIndexer
modelo_idx = indexer_airline.fit(df_clean)
df_ejemplo = modelo_idx.transform(df_clean)
df_ejemplo.select("AIRLINE", "airline_idx").distinct().orderBy("airline_idx").show()

## 4. Feature Engineering: Binarizer

`Binarizer` convierte una columna numerica en binaria (0 o 1) segun un umbral. Usaremos un umbral de 15 minutos para definir "retrasado".

In [ ]:
# Crear variable target binaria: 1 si DEPARTURE_DELAY > 15, 0 si no
binarizer = Binarizer(threshold=15.0, inputCol="DEPARTURE_DELAY", outputCol="label")

# Ejemplo: ver el resultado del Binarizer
df_bin = binarizer.transform(df_clean)
df_bin.select("DEPARTURE_DELAY", "label").show(10)

# Distribucion de la variable target
df_bin.groupBy("label").count().show()

## 5. Feature Engineering: VectorAssembler

`VectorAssembler` combina multiples columnas en un unico vector de features, que es el formato requerido por los modelos de MLlib.

In [ ]:
# Combinar features en un solo vector
assembler = VectorAssembler(
    inputCols=["airline_idx", "origin_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"],
    outputCol="features"
)

print("Features que se usaran:")
print("  - airline_idx: aerolinea codificada")
print("  - origin_idx: aeropuerto origen codificado")
print("  - MONTH: mes del vuelo")
print("  - DAY_OF_WEEK: dia de la semana")
print("  - DISTANCE: distancia del vuelo")

## 6. Division Train/Test

Dividimos los datos en 80% entrenamiento y 20% prueba. Usamos `seed=42` para reproducibilidad.

In [ ]:
# Dividir datos en train y test
train, test = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Datos de entrenamiento: {train.count()}")
print(f"Datos de prueba: {test.count()}")

## 7. Modelo: Decision Tree Classifier

El arbol de decision es un modelo interpretable que divide los datos en nodos basandose en las features. `maxDepth` controla la profundidad maxima del arbol.

In [ ]:
# Crear clasificador de arbol de decision
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=5)

print("Modelo: DecisionTreeClassifier")
print(f"  maxDepth: {dt.getMaxDepth()}")
print(f"  labelCol: {dt.getLabelCol()}")
print(f"  featuresCol: {dt.getFeaturesCol()}")

## 8. Construir y Entrenar Pipeline

El Pipeline encadena todos los pasos: codificacion, binarizacion, ensamblado de features y clasificador. Con un solo `fit()` se entrena todo.

In [ ]:
# Construir pipeline completo
pipeline = Pipeline(stages=[
    indexer_airline,   # Codificar AIRLINE
    indexer_origin,    # Codificar ORIGIN_AIRPORT
    binarizer,         # Crear label binaria
    assembler,         # Combinar features
    dt                 # Clasificador
])

print("Pipeline creado con 5 etapas:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  {i+1}. {type(stage).__name__}")

In [ ]:
# Entrenar el pipeline
print("Entrenando pipeline...")
model = pipeline.fit(train)
print("Pipeline entrenado exitosamente!")

In [ ]:
# Generar predicciones sobre datos de prueba
predictions = model.transform(test)

# Ver resultados
predictions.select("AIRLINE", "ORIGIN_AIRPORT", "DEPARTURE_DELAY", "label", "prediction", "probability").show(10)

## 9. Evaluacion del Modelo

Usamos `MulticlassClassificationEvaluator` para medir la precision (accuracy) del modelo y `BinaryClassificationEvaluator` para el area bajo la curva ROC.

In [ ]:
# Evaluar accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

# Evaluar precision y recall
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)
precision = evaluator_precision.evaluate(predictions)
recall = evaluator_recall.evaluate(predictions)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
# Evaluar AUC (Area Under ROC Curve)
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(predictions)
print(f"AUC-ROC: {auc:.4f}")

In [ ]:
# Matriz de confusion
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

In [ ]:
# Importancia de features (solo para arboles de decision)
dt_model = model.stages[-1]  # Ultimo stage es el clasificador entrenado
feature_names = ["airline_idx", "origin_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"]

print("Importancia de Features:")
print("-" * 35)
for name, importance in zip(feature_names, dt_model.featureImportances.toArray()):
    print(f"  {name:20s}: {importance:.4f}")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Agregar mas features al modelo
# =============================================================
# TODO: Mejora el modelo agregando DESTINATION_AIRPORT como feature.
#
# Pasos:
#   1. Crear un StringIndexer para DESTINATION_AIRPORT:
#      indexer_dest = StringIndexer(inputCol="DESTINATION_AIRPORT", outputCol="dest_idx")
#
#   2. Crear un nuevo VectorAssembler que incluya "dest_idx":
#      assembler_v2 = VectorAssembler(
#          inputCols=["airline_idx", "origin_idx", "dest_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"],
#          outputCol="features"
#      )
#
#   3. Crear nuevo pipeline con el indexer adicional y el nuevo assembler
#   4. Entrenar y evaluar
#   5. Comparar accuracy con el modelo anterior
#
# Pista: El pipeline debe tener 6 stages ahora
#   (3 indexers + binarizer + assembler + clasificador)

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Random Forest Classifier
# =============================================================
# TODO: Reemplaza el DecisionTreeClassifier por un
#   RandomForestClassifier y compara los resultados.
#
# Pasos:
#   1. Crear RandomForestClassifier:
#      rf = RandomForestClassifier(
#          labelCol="label", 
#          featuresCol="features", 
#          numTrees=20
#      )
#
#   2. Crear nuevo pipeline reemplazando dt por rf
#   3. Entrenar con train y predecir con test
#   4. Evaluar accuracy, precision, recall y AUC
#   5. Comparar con el DecisionTree:
#      - Cual tiene mejor accuracy?
#      - Cual tiene mejor AUC?
#
# Pista: RandomForestClassifier ya esta importado arriba

# Escribe tu codigo aqui:


---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Busqueda de hiperparametros con CrossValidator.

In [ ]:
# =============================================================
# DESAFIO: CrossValidator para busqueda de hiperparametros
# =============================================================
# TODO: Usa CrossValidator para encontrar la mejor combinacion
#   de hiperparametros para RandomForestClassifier.
#
# Pasos:
#   1. Importar:
#      from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
#
#   2. Crear el pipeline base (sin clasificador):
#      pipeline_base = Pipeline(stages=[indexer_airline, indexer_origin, binarizer, assembler])
#      df_prepared = pipeline_base.fit(train).transform(train)
#      df_test_prepared = pipeline_base.fit(train).transform(test)
#
#   3. Crear RandomForestClassifier:
#      rf = RandomForestClassifier(labelCol="label", featuresCol="features")
#
#   4. Crear grid de parametros:
#      paramGrid = ParamGridBuilder() \
#          .addGrid(rf.maxDepth, [3, 5, 7, 10]) \
#          .addGrid(rf.numTrees, [10, 20, 50]) \
#          .build()
#
#   5. Crear CrossValidator:
#      cv = CrossValidator(
#          estimator=rf,
#          estimatorParamMaps=paramGrid,
#          evaluator=BinaryClassificationEvaluator(labelCol="label"),
#          numFolds=3
#      )
#
#   6. Entrenar: cvModel = cv.fit(df_prepared)
#   7. Obtener mejor modelo: cvModel.bestModel
#   8. Reportar: mejor maxDepth y numTrees
#      print(cvModel.bestModel.getMaxDepth())
#      print(cvModel.bestModel.getNumTrees)
#
# NOTA: Esto puede tardar varios minutos en ejecutarse.

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **MLlib:** Libreria de Machine Learning distribuida de Spark
2. **StringIndexer:** Codificar variables categoricas a indices numericos
3. **Binarizer:** Crear variable target binaria a partir de un umbral
4. **VectorAssembler:** Combinar multiples columnas en un vector de features
5. **Pipeline:** Encadenar transformaciones y modelo en un flujo reproducible
6. **DecisionTreeClassifier:** Modelo de arbol de decision para clasificacion
7. **RandomForestClassifier:** Ensemble de arboles para mejor precision
8. **Evaluadores:** Accuracy, precision, recall y AUC para medir rendimiento
9. **CrossValidator:** Busqueda sistematica de hiperparametros optimos

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")